# CAWOT-CM V0 — Kaggle end-to-end (budget sweep)

**Source of truth: HuggingFace** [`TruongVox/Cawot-dataset`](https://huggingface.co/datasets/TruongVox/Cawot-dataset).

**V0 plan (clean baseline, no qproxy):**
1. Download N shards from HF, extract
2. Random sample 50K (image, caption) entries
3. Extract CLIP ViT-B/16 embeddings
4. Hold out 5K pairs as a FIXED retrieval val set (bigger gallery → less saturation than 2K)
5. **Budget sweep {5, 10, 20, 40}%**: for each — cluster, select Random vs V0 (farthest-from-centroid), fine-tune CLIP-B/16, eval
6. Plot R@1 vs budget — V0 should beat Random most at low budget, converge at high

**Setup (Kaggle UI):** GPU P100 + Internet ON. No external dataset needed.

## 1. Env

In [ ]:
!nvidia-smi -L
!df -h /kaggle/working | tail -1

## 2. Clone repo + install deps

In [ ]:
import os
if not os.path.exists("/kaggle/working/cawot-cm"):
    !git clone https://github.com/HohoHocCode/cawot-cm.git /kaggle/working/cawot-cm
%cd /kaggle/working/cawot-cm
!pip install -q open_clip_torch faiss-gpu-cu12 einops huggingface_hub

## 3. Download N shards from HuggingFace

Each shard ≈ 1.4 GB zip + ~5 MB JSON. 5 shards ≈ 7 GB download, ~14 GB extracted.

In [ ]:
NUM_SHARDS = 5
!python scripts/setup_data.py --output /kaggle/working/pab_data --num-shards {NUM_SHARDS}

## 4. Sanity check that one image resolves

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/cawot-cm")
from src.data import build_pool, TrainPoolDataset
from torchvision import transforms

anns, shard_roots = build_pool(
    annotations_dir="/kaggle/working/pab_data/annotations",
    image_root="/kaggle/working/pab_data/images",
    sample_size=100, seed=42,
)
print(f"loaded {len(anns)} annotations; shards: {sorted(shard_roots)}")
tx = transforms.Compose([transforms.Resize(224), transforms.CenterCrop(224), transforms.ToTensor()])
ds = TrainPoolDataset(anns, shard_roots, image_transform=tx)
s = ds[0]
print(f"image shape={s['image'].shape}; caption: {s['caption'][:80]}...")
print("\u2713 images resolve correctly")

## 5. Run V0 budget sweep

Default config: budgets {5,10,20,40}%, 1 seed. On P100:
- Extract embeddings (50K): ~15-20 min (cached after first run)
- 4 budgets × 2 methods × 1 seed = 8 fine-tune runs: ~35-45 min total
- 8 evals + 1 zeroshot (5K gallery each): ~20 min
- **Total ~70-80 min for 1 seed**

After confirming the eval de-saturates and V0 separates from Random, bump `train.seeds` to `[42, 1, 2]` in config.yaml for error bars (~2.5h).

In [ ]:
!python scripts/run_v0.py --config config.yaml

## 6. Results table

In [ ]:
import json, pandas as pd
with open("/kaggle/working/outputs/eval/summary.json") as f:
    summary = json.load(f)
print("zeroshot mean_R@1:", summary["zeroshot"]["mean_R@1"])

rows = []
for method in ["random", "v0"]:
    for budget, stat in summary[method].items():
        rows.append({"method": method, "budget": float(budget),
                     "mean_R@1": stat["mean_R@1_mean"], "std": stat["mean_R@1_std"]})
df = pd.DataFrame(rows).pivot(index="budget", columns="method", values="mean_R@1")
df["delta(v0-rand)"] = df["v0"] - df["random"]
df

## 7. Plot R@1 vs budget

In [ ]:
import matplotlib.pyplot as plt

budgets = sorted(float(b) for b in summary["random"].keys())
rand = [summary["random"][str(b)]["mean_R@1_mean"] for b in budgets]
rand_e = [summary["random"][str(b)]["mean_R@1_std"] for b in budgets]
v0 = [summary["v0"][str(b)]["mean_R@1_mean"] for b in budgets]
v0_e = [summary["v0"][str(b)]["mean_R@1_std"] for b in budgets]
zs = summary["zeroshot"]["mean_R@1"]

x = [b * 100 for b in budgets]
plt.figure(figsize=(7, 5))
plt.errorbar(x, rand, yerr=rand_e, marker="o", label="Random", capsize=3)
plt.errorbar(x, v0, yerr=v0_e, marker="s", label="V0 (farthest-from-centroid)", capsize=3)
plt.axhline(zs, ls="--", c="gray", label=f"zero-shot ({zs:.1f})")
plt.xlabel("Budget (% of train pool)")
plt.ylabel("mean R@1 (val image-text retrieval)")
plt.title("V0 budget sweep")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("/kaggle/working/outputs/eval/budget_curve.png", dpi=120, bbox_inches="tight")
plt.show()

## 8. What to look for

- **Eval de-saturated?** With the 5K gallery, R@1 should be well below the old 94% ceiling — ideally 70-90% with visible headroom. If still ~99%, increase `data.val_size` further.
- **V0 vs Random**: expect the gap `Δ(v0-rand)` to be **largest at 5%** and shrink toward 0 at 40%. That's the coreset story: selection matters most under tight budget.
- **If V0 ≤ Random at all budgets** even after de-saturation: farthest-from-centroid is genuinely weak (it picks outliers). That's a real finding → motivates V1 (facility location picks representative samples).
- **Single seed = no error bars.** Bump `train.seeds: [42, 1, 2]` once the trend looks right.

**Save Version → Save & Run All** to persist `outputs/` (summary.json, records.csv, budget_curve.png).